# Model Comparison: TF-IDF, SVM, and BiLSTM

This notebook compares three emotion-classification baselines on the processed dataset:

- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM
- BiLSTM

It uses the existing train/valid/test splits from `data/processed/` and saves comparison outputs under `reports/model_comparison/`.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn tensorflow

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('TensorFlow version:', tf.__version__)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'model_comparison'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
valid_df = pd.read_csv(DATA_DIR / 'valid.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

display(train_df.head())
print('Train:', train_df.shape)
print('Valid:', valid_df.shape)
print('Test :', test_df.shape)
print('\nLabels:', sorted(train_df['final_emotion'].unique()))

In [ ]:
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['final_emotion'].astype(str).tolist()

X_valid = valid_df['text'].astype(str).tolist()
y_valid = valid_df['final_emotion'].astype(str).tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['final_emotion'].astype(str).tolist()

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_valid_enc = label_encoder.transform(y_valid)
y_test_enc  = label_encoder.transform(y_test)

label_names = label_encoder.classes_.tolist()
num_classes = len(label_names)
print('Encoded labels:', dict(zip(label_names, range(num_classes))))

## Helper Functions

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


def save_confusion_matrix(y_true, y_pred, labels, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


results = []
detailed_reports = {}

## 1. TF-IDF + Logistic Regression

In [ ]:
tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words='english')),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=None))
])

tfidf_lr.fit(X_train, y_train)
lr_pred = tfidf_lr.predict(X_test)

results.append(compute_metrics(y_test, lr_pred, 'TF-IDF + Logistic Regression'))
detailed_reports['TF-IDF + Logistic Regression'] = classification_report(y_test, lr_pred, output_dict=True, zero_division=0)

print(classification_report(y_test, lr_pred, zero_division=0))
save_confusion_matrix(y_test, lr_pred, label_names, 'TF-IDF + Logistic Regression', REPORT_DIR / 'cm_tfidf_logreg.png')

## 2. TF-IDF + Linear SVM

In [ ]:
tfidf_svm = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words='english')),
    ('clf', LinearSVC(class_weight='balanced'))
])

tfidf_svm.fit(X_train, y_train)
svm_pred = tfidf_svm.predict(X_test)

results.append(compute_metrics(y_test, svm_pred, 'TF-IDF + Linear SVM'))
detailed_reports['TF-IDF + Linear SVM'] = classification_report(y_test, svm_pred, output_dict=True, zero_division=0)

print(classification_report(y_test, svm_pred, zero_division=0))
save_confusion_matrix(y_test, svm_pred, label_names, 'TF-IDF + Linear SVM', REPORT_DIR / 'cm_tfidf_svm.png')

## 3. BiLSTM

In [ ]:
MAX_WORDS = 30000
MAX_LEN = 120
EMBED_DIM = 128
BATCH_SIZE = 64
EPOCHS = 10

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_valid_seq = tokenizer.texts_to_sequences(X_valid)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_valid_pad = pad_sequences(X_valid_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

y_train_cat = to_categorical(y_train_enc, num_classes=num_classes)
y_valid_cat = to_categorical(y_valid_enc, num_classes=num_classes)
y_test_cat  = to_categorical(y_test_enc, num_classes=num_classes)

bilstm_model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

bilstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = bilstm_model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_valid_pad, y_valid_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
bilstm_probs = bilstm_model.predict(X_test_pad)
bilstm_pred_enc = bilstm_probs.argmax(axis=1)
bilstm_pred = label_encoder.inverse_transform(bilstm_pred_enc)

results.append(compute_metrics(y_test, bilstm_pred, 'BiLSTM'))
detailed_reports['BiLSTM'] = classification_report(y_test, bilstm_pred, output_dict=True, zero_division=0)

print(classification_report(y_test, bilstm_pred, zero_division=0))
save_confusion_matrix(y_test, bilstm_pred, label_names, 'BiLSTM', REPORT_DIR / 'cm_bilstm.png')

## Compare Results

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1_weighted', ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=results_df, x='model', y='accuracy', palette='viridis', ax=ax)
ax.set_title('Model Accuracy Comparison')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.tick_params(axis='x', rotation=12)
for i, row in results_df.iterrows():
    ax.text(i, row['accuracy'] + 0.005, f"{row['accuracy']:.3f}", ha='center', fontsize=9)
plt.tight_layout()
plot_path = REPORT_DIR / 'model_accuracy_comparison.png'
fig.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved plot to:', plot_path)

In [ ]:
results_path = REPORT_DIR / 'model_comparison_results.csv'
results_df.to_csv(results_path, index=False)

reports_path = REPORT_DIR / 'detailed_classification_reports.json'
with open(reports_path, 'w') as f:
    json.dump(detailed_reports, f, indent=2)

print('Saved results table to:', results_path)
print('Saved detailed reports to:', reports_path)

## Notes for Report

- Use `model_comparison_results.csv` in your final report table.
- Use the confusion matrices to discuss class-wise strengths and weaknesses.
- Compare this notebook's best model with your RoBERTa results from notebook 03.